---

- Author: Jaelin Lee
- Date: Feb 7, 2026
- Description: creates a cleaned session log for a single session log.
- Output: saves a cleaned session_log to **app/logs/cleaned/** folder
- Example: `app/logs/cleaned/cleaned_session_orpda_20260207_180031_gpt-oss:20b-cloud_0.0_hailey.csv`

---

## 0. Import

In [260]:
import os
from pathlib import Path
from pprint import pprint
from random import random

import pandas as pd
from warnings import filterwarnings
filterwarnings("ignore")

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

## 1. Load memory_streams.log file (JSONL)

In [261]:
ROOT = Path.cwd().parents[0]
LOGS_PATH = Path(ROOT, 'app/logs/')
LOGS_PATH

PosixPath('/Users/jaelinlee/Driftville_Agent/app/logs')

In [262]:
#########################################
#>> UPDATE INPUT
# Option 1 - old runs
n = 1
path = Path(LOGS_PATH, "v3_complete_waking_hours") # folder name of log files
mode = "orpda"  # "orpda", "orpa", None

# Option 2 - current runs
# path = Path(LOGS_PATH, "") # folder name of log files
# mode = None
#########################################

# Extract log file names
if mode == "orpa":
    sessions = sorted([f.name for f in path.glob("session_orpa_*") if "memory" not in f.name], key=lambda x: Path(path / x).stat().st_birthtime, reverse=True)
elif mode == "orpda": 
    sessions = sorted([f.name for f in path.glob("session_orpda*") if "memory" not in f.name], key=lambda x: Path(path / x).stat().st_birthtime, reverse=True)
else:
    sessions = sorted([f.name for f in path.glob("*") if "memory" not in f.name], key=lambda x: Path(path / x).stat().st_birthtime, reverse=True)
    print(sessions)
    if sessions:
        mode = "orpda" if "orpda" in sessions[0] else "orpa" if "orpa" in sessions[0] else None


# Select the latest file
if path == Path(LOGS_PATH, ""):
    print(sessions)
    session_path = Path(path, sessions[0]) if len(sessions)>0 else Path(path,  sessions)
else:
    session_path = Path(path, sessions[n]) if len(sessions)>0 else Path(path,  sessions)


print("\n", "="*10, mode.upper(), "="*10)
print(len(sessions), "logs")
pprint(sessions)
print("\n", "="*10, "Selected File", "="*10)
print(session_path)


 ========== ORPDA ==========
14 logs
['session_orpda_20260207_180031_gpt-oss:20b-cloud_0.0_hailey.log',
 'session_orpda_20260207_171847_gpt-oss:20b-cloud_1.0_hailey.log',
 'session_orpda_20260207_145217_cogito-2.1:671b-cloud_0.3_hailey.log',
 'session_orpda_20260207_135027_cogito-2.1:671b-cloud_0.8_hailey.log',
 'session_orpda_20260207_131424_cogito-2.1:671b-cloud_0.0_hailey.log',
 'session_orpda_20260207_124952_cogito-2.1:671b-cloud_1.0_hailey.log',
 'session_orpda_20260207_122012_cogito-2.1:671b-cloud_1.0_hailey.log',
 'session_orpda_20260207_105208_cogito-2.1:671b-cloud_0.0_hailey.log',
 'session_orpda_20260207_083210_gemma3:27b-cloud_0.0_hailey.log',
 'session_orpda_20260207_080202_gemma3:27b-cloud_0.8_hailey.log',
 'session_orpda_20260206_193105_gemini-3-flash-preview:cloud_0.0_hailey.log',
 'session_orpda_20260206_171410_gemini-3-flash-preview:cloud_0.8_hailey.log',
 'session_orpda_20260205_225125_gemini-3-flash-preview:cloud_0.3_hailey.log',
 'session_orpda_20260205_211529_gemi

## 2. Utils

In [263]:
# def load_memory_stream(f_path):
#     df = pd.read_json(open(f_path), lines=True)

#     # Extract the action summary of the Action layer
#     df["action_summary"] = df["summary"].apply(lambda x: x.split(';')[-1])
#     df["sim_time"] = df["sim_time"].astype(str).str.extract(r'(\d{2}:\d{2})')
#     df["llm_temperature"] = df["llm_temperature"].astype(float).round(1).astype(str)
#     df.rename(columns={"llm_temperature": "temp"}, inplace=True)

#     # format DF
#     df_final = df[['sim_time', 'agent', 'temp', 'action_summary']]
#     return df_final

# # load_memory_stream(stream_path).style.set_properties(**{'text-align': 'left'})
# # df_stream = load_memory_stream(memory_path)
# # df_stream.head(3)

In [264]:
# def load_sessions(path, mode=None):
#     """Load session files from path, optionally filtered by mode."""
#     if mode == "orpa":
#         pattern = "session_orpa_*"
#     elif mode == "orpda":
#         pattern = "session_orpda*"
#     else:
#         pattern = "*"
    
#     sessions = sorted(
#         [f.name for f in path.glob(pattern) if "memory" not in f.name],
#         key=lambda x: Path(path / x).stat().st_birthtime,
#         reverse=True
#     )
    
#     # Auto-detect mode if not specified
#     if mode is None and sessions:
#         mode = "orpda" if "orpda" in sessions[0] else "orpa" if "orpa" in sessions[0] else None
    
#     return sessions, mode

# # Usage
# path = Path("v3_complete_waking_hours")
# path = ""
# sessions, mode = load_sessions(path, mode="orpa")
# session_path = Path(path, sessions[0]) if sessions else None

# print(f"\n{'='*10} Selected File {'='*10}")
# print(session_path)
# print(f"\n{'='*10} {mode.upper() if mode else 'UNKNOWN'} {'='*10}")
# pprint(sessions)

In [265]:
def load_session_log(f_path):
    df = pd.read_json(f_path, lines=True)
    df = df.convert_dtypes() 
    
    # Extract the action summary of the Action layer
    df["sim_time"] = df["sim_time"].astype(str).str.extract(r'(\d{2}:\d{2})')
    df["llm_temperature"] = df["llm_temperature"].astype(float).round(1).astype(str)
    df.rename(columns={"llm_temperature": "temp"}, inplace=True)
    df.drop(columns=["ts_created"], inplace=True)      
    display(df.head(3))
       
    # print keys from jsonl  
    keys = df["orpda"][0].keys()
    print(keys)
    df_final = pd.concat([df.drop("orpda", axis=1), df["orpda"].apply(pd.Series)], axis=1)    
    return df_final

df_session = load_session_log(session_path).dropna(subset=['llm_model'])
df_session.tail()
df_session.shape

,llm_model,temp,tick,sim_time,agent,use_drift,orpda
0,gpt-oss:20b-cloud,1.0,0,10:00,Hailey Johnson,True,"{'observation': {'datetime_start': '2023-02-13 10:00', 'duration_min': 15, 'location': 'home:bathroom', 'action': 'morning_routine', 'environment_description': 'splashing water, scent of peppermint, phone buzzing with social media alerts, bright morning light', 'recent_history': [], 'state_summary': 'Hailey Johnson is at home:bathroom doing morning_routine.', 'topic': 'Hailey wakes up feeling refreshed.'}, 'reflection': {'plan_alignment': 'aligned', 'boredom_fatigue': 'low', 'attention_stability': 'stable', 'rumination_level': 'none', 'rumination_theme': None, 'emotional_residue': 'none', 'emerging_thought_pattern': None, 'competing_stimuli': ['phone notifications', 'bright morning light'], 'executive_insight': '', 'meta_rule': 'continue', 'state_summary': 'Hailey Johnson is focused on morning routine, low distraction observed.', 'reasoning': 'No evidence of drift or fatigue; plan likely on track.'}, 'plan': {'location': 'home:bathroom', 'action': 'morning_routine', 'datetime_start': '2023-02-13 10:00', 'duration_min': 15, 'topic': 'Hailey wakes up feeling refreshed.', 'state_summary': 'Heat coffee, set alarm, review tomorrow’s tasks, 5‑minute stretch.'}, 'drift_decision': {'should_drift': False, 'drift_type': 'none', 'drift_topic': '', 'drift_action': 'continue', 'drift_intensity': 0, 'potential_recovery': '', 'justification': '', 'datetime_start': '2023-02-13 10:00', 'duration_min': 15}, 'action_result': {'datetime_start': '2023-02-13 10:00', 'duration_min': 15, 'location': 'home:bathroom', 'action': 'morning_routine', 'topic': 'Hailey wakes up feeling refreshed.', 'should_drift': False, 'drift_type': 'none', 'drift_intensity': 0.0, 'state_summary': 'Heat coffee, set alarm, review tomorrow’s tasks, 5‑minute stretch.', 'drift_topic': '', 'next_datetime': '2023-02-13 10:15'}}"
1,gpt-oss:20b-cloud,1.0,1,10:15,Hailey Johnson,True,"{'observation': {'datetime_start': '2023-02-13 10:15', 'duration_min': 15, 'location': 'home:bathroom', 'action': 'morning_routine', 'environment_description': 'splashing water, scent of peppermint, phone buzzing with social media alerts, bright morning light', 'recent_history': [{'sim_time': '2023-02-13 10:00', 'summary': 'Hailey Johnson is at home:bathroom doing morning_routine about Hailey wakes up feeling refreshed. ; (Hailey Johnson is focused on morning routine, low distraction observed.) ; Heat coffee, set alarm, review tomorrow’s tasks, 5‑minute stretch.'}], 'state_summary': 'Hailey Johnson is at home:bathroom doing morning_routine.'}, 'reflection': {'plan_alignment': 'aligned', 'boredom_fatigue': 'low', 'attention_stability': 'stable', 'rumination_level': 'none', 'rumination_theme': None, 'emotional_residue': 'low', 'emerging_thought_pattern': None, 'competing_stimuli': ['phone social media alerts', 'splashing water', 'peppermint scent', 'bright morning light'], 'executive_insight': '', 'meta_rule': 'continue', 'state_summary': 'Hailey is focused on a calm, productive morning routine with minimal distractions.', 'reasoning': 'No drift signals detected; attention is stable and plan aligns with routine.'}, 'plan': {'location': 'home:bathroom', 'action': 'morning_routine', 'datetime_start': '2023-02-13 10:15', 'duration_min': 15, 'topic': 'Hailey wakes up feeling refreshed.', 'state_summary': 'Hailey takes a calm 15‑minute break to brew coffee, stretch, and plan the day ahead.'}, 'drift_decision': {'should_drift': False, 'drift_type': 'none', 'drift_topic': '', 'drift_action': 'continue', 'drift_intensity': 0, 'potential_recovery': '', 'justification': '', 'datetime_start': '2023-02-13 10:15', 'duration_min': 15}, 'action_result': {'datetime_start': '2023-02-13 10:15', 'duration_min': 15, 'location': 'home:bathroom', 'action': 'morning_routine', 'topic': 'Hailey wakes up feeling refreshed.', 'should_drift': False, 'drift_type': 'none', 'drift_intensity': 0.0, 'state_summary': 'Hailey takes a c

dict_keys(['observation', 'reflection', 'plan', 'drift_decision', 'action_result'])


(65, 11)

In [266]:
def get_df_name(df):
    for name, obj in globals().items():
        if obj is df:
            return name
    return None

## 3. Convert JSONL content to Series

In [267]:
# df_session["location_plan"] = df_session["plan"].apply(lambda x: x["location"])
# df_session["location_action"] = df_session["action_result"].apply(lambda x: x["location"])

# Extract ORPDA layer to each dataframe
df_observe = df_session["observation"].apply(pd.Series)
df_observe.head(2)

df_plan = df_session["plan"].apply(pd.Series)
df_plan.head(2)

if mode=="orpda":
    df_drift = df_session["drift_decision"].apply(pd.Series)
    df_drift.head(2)

df_act = df_session["action_result"].apply(pd.Series)
df_act.head(2)

df_reflect = df_session["reflection"].apply(pd.Series)
df_reflect.head(2)


,plan_alignment,boredom_fatigue,attention_stability,rumination_level,rumination_theme,emotional_residue,emerging_thought_pattern,competing_stimuli,executive_insight,meta_rule,state_summary,reasoning,0
0,aligned,low,stable,none,None,none,None,"[phone notifications, bright morning light]",,continue,"Hailey Johnson is focused on morning routine, low distraction observed.",No evidence of drift or fatigue; plan likely on track.,NaN
1,aligned,low,stable,none,None,low,None,"[phone social media alerts, splashing water, peppermint scent, bright morning light]",,continue,"Hailey is focused on a calm, productive morning routine with minimal distractions.",No drift signals detected; attention is stable and plan aligns with routine.,NaN


In [268]:
# check for alignment
cols_to_check = ["datetime_start", "location", "action"]
print("Mismatch count")

# Selct comparison
key1 = df_observe
key2 = df_plan
print("=" * 20)
print(f"{get_df_name(key1)} vs {get_df_name(key2)} (#rows: {len(df_observe)})")
print("=" * 20)
# Count mismatch
for col in cols_to_check:
    print(f"{col}: ", (key1[col] != key2[col]).sum())
    
# Selct comparison
key1 = df_observe
key2 = df_act
print("=" * 20)
print(f"{get_df_name(key1)} vs {get_df_name(key2)} (#rows: {len(df_observe)})")
print("=" * 20)  
# Count mismatch
for col in cols_to_check:
    print(f"{col}: ", (key1[col] != key2[col]).sum())
    
# Selct comparison
key1 = df_plan
key2 = df_act
print("=" * 20)
print(f"{get_df_name(key1)} vs {get_df_name(key2)} (#rows: {len(df_observe)})")
print("=" * 20)
# Count mismatch
for col in cols_to_check:
    print(f"{col}: ", (key1[col] != key2[col]).sum())


Mismatch count
df_observe vs df_plan (#rows: 65)
datetime_start:  1
location:  11
action:  10
df_observe vs df_act (#rows: 65)
datetime_start:  1
location:  12
action:  9
df_plan vs df_act (#rows: 65)
datetime_start:  2
location:  2
action:  3


## 4. Filter ORPDA columns

In [269]:

# O
o_df = df_observe[['datetime_start', 'location', 'action', 'state_summary', 'environment_description']]
o_df.columns = o_df.columns + "_o"
# o_df.rename(columns={"datetime_start_o": "datetime_start"}, inplace=True)

# P
p_df = df_plan[['datetime_start', 'location', 'action', 'topic', 'state_summary']]
p_df.columns = p_df.columns + "_p"
# p_df.rename(columns={"datetime_start_p": "datetime_start"}, inplace=True)

# D
if mode == "orpda":
    d_df = df_drift.drop(columns=["duration_min", "drift_intensity"])
    d_df = d_df[['datetime_start', 'should_drift', 'drift_type', 'drift_topic', 'drift_action', 'potential_recovery', 'justification']]
    d_df.columns = d_df.columns + "_d"
    # d_df.rename(columns={"datetime_start_d": "datetime_start"}, inplace=True)

# A
a_df = df_act[['datetime_start','location', 'action', 'topic', 'drift_type', 'drift_topic', 'state_summary']]
a_df.columns = a_df.columns + "_a"
# a_df.rename(columns={"datetime_start_a": "datetime_start"}, inplace=True)


# R
df_reflect["datetime_start"] = a_df["datetime_start_a"].copy()
r_df = df_reflect[['datetime_start', 'rumination_theme', 'emerging_thought_pattern', 'executive_insight', 'state_summary', 'reasoning', 'meta_rule']]

r_df.columns = r_df.columns + "_r"
# r_df.rename(columns={"datetime_start_r": "datetime_start"}, inplace=True)


# Display DF
print("Observe:")
# print(o_df.columns)
display(o_df.tail())
print("Reflect:")
display(r_df.tail())
print("Plan:")
display(p_df.tail())
if mode == "orpda":
    print("Drift:")
    display(d_df.tail())
    print("Act:")
display(a_df.tail())


Observe:


,datetime_start_o,location_o,action_o,state_summary_o,environment_description_o
60,2023-02-14 01:00,writer_desk,writing,Hailey Johnson is at writer_desk doing writing.,"clicking of keys, hum of the desk lamp, phone pings with messages, cool night air"
61,2023-02-14 01:15,writer_desk,writing,Hailey Johnson is at writer_desk doing writing.,"clicking of keys, hum of the desk lamp, phone pings with messages, cool night air"
62,2023-02-14 01:30,writer_desk,writing,Hailey Johnson is at writer_desk doing writing.,"running water, scent of face wash, soft towel, phone screen dimming"
63,2023-02-14 01:45,home:bathroom,night_routine,Hailey Johnson is at home:bathroom doing night_routine.,"running water, scent of face wash, soft towel, phone screen dimming"
64,2023-02-14 02:00,home:bathroom,night_routine,Hailey Johnson is at home:bathroom doing night_routine.,{}


Reflect:


,datetime_start_r,rumination_theme_r,emerging_thought_pattern_r,executive_insight_r,state_summary_r,reasoning_r,meta_rule_r
60,2023-02-14 01:00,podcast brainstorming,repeated shift to podcast ideas while writing,Redirect focus to finishing the writing before returning to podcast ideas,"Hailey writing but repeatedly drifting toward podcast brainstorming, showing fatigue and fragile focus",Fatigue drives ongoing drift away from the primary writing task.,reset_plan
61,NaN,podcast brainstorming,recurrent distraction by podcast ideas,Encourage short breaks and set a timer to focus on writing.,"Hailey is writing but repeatedly distracted by podcast brainstorming, showing fatigue.",Frequent topic drift and fatigue signal fragile focus; reset plan to refocus on writing.,reset_plan
62,2023-02-14 01:30,podcast brainstorming,persistent podcast idea loop,"Recenter on writing outline, limit podcast notes.","Hailey drifting from writer task to podcast ideas, showing fatigue and fragile focus.","Observation shows repeated leak toward podcast brainstorming, indicating high fatigue, suboptimal attention, requiring plan reset.",reset_plan
63,2023-02-14 01:45,NaN,NaN,NaN,NaN,NaN,NaN
64,2023-02-14 02:00,NaN,NaN,NaN,NaN,NaN,NaN


Plan:


,datetime_start_p,location_p,action_p,topic_p,state_summary_p
60,2023-02-14 01:00,writer_desk,writing,Another late night writing session.,Begin focused writing session to finish the current draft.
61,2023-02-14 01:15,writer_desk,writing,Another late night writing session.,Take a quick break to regain focus.
62,2023-02-14 01:30,home:bathroom,night_routine,Preparing for a 2am bedtime.,Shift to light admin work to reset focus before resuming writing.
63,2023-02-14 01:45,home:bathroom,night_routine,Preparing for a 2am bedtime.,Admin tasks to calm fatigue before resuming deeper work.
64,2023-02-14 02:00,home:bedroom,night_routine,complete a short writing task to ground focus,Switch to gentle writing during night routine to regain attention before sleep.


Drift:


,datetime_start_d,should_drift_d,drift_type_d,drift_topic_d,drift_action_d,potential_recovery_d,justification_d
60,2023-02-14 01:00,True,internal,podcast brainstorming,momentary pause to jot a quick outline,set a 5‑min timer to refocus writing,"Hailey’s fatigue pulls her into podcast ideas, disrupting the draft and inviting a brief pause."
61,2023-02-14 01:15,True,attentional_leak,podcast brainstorming,continue,Take a 5‑minute break or set a timer to re‑anchor writing focus.,"Hailey keeps wrestling with podcast ideas, but fatigue keeps focus fraying; short break could help."
62,2023-02-14 01:30,True,internal,podcast brainstorming,pause wiping face to mentally note episode ideas,Quickly resume towel and wash routine after jotting brief notes.,"Hailey hits speaker from podcast thoughts, delaying a wash moment due to fatigue and persistent ideas."
63,2023-02-14 01:45,True,internal,podcast brainstorming,continue,"Briefly pause, look at phone’s calendar; a reminder of the admin task could anchor her focus back upon waking.","Hailey’s fatigue keeps podcast ideas looping, dragging her mind from the moist towel toward episode outlines."
64,2023-02-14 02:00,True,internal,podcast brainstorming,continue night routine,5‑minute mindful breathing to reclaim focus,"Hailey’s sheer fatigue and restless podcast ideas hijack her routine, leaving her mentally rummaging."


Act:


,datetime_start_a,location_a,action_a,topic_a,drift_type_a,drift_topic_a,state_summary_a
60,2023-02-14 01:00,writer_desk,writing,Another late night writing session.,internal,podcast brainstorming,"Writing on the novel outline at home:writer_desk, while mind drifts to podcast brainstorming"
61,NaN,NaN,NaN,NaN,NaN,NaN,NaN
62,2023-02-14 01:30,home:bathroom,night_routine,Preparing for a 2am bedtime.,internal,podcast brainstorming,Wiping face in bathroom while mind drifts to podcast brainstorming.
63,2023-02-14 01:45,home:bathroom,night_routine,Preparing for a 2am bedtime.,internal,podcast brainstorming,Continue writing content outline in home:studio while mind drifts to podcast brainstorming.
64,2023-02-14 02:00,home:bedroom,night_routine,sleep prep,internal,podcast brainstorming,Completed bedtime prep at home:bedroom while mind drifts to podcast brainstorming


## 5. Merge ORPDA with selected columns

In [270]:
# # Merge 
# tmp = pd.merge(o_df, r_df, on="datetime_start", how="outer")
# tmp = pd.merge(tmp, p_df, on="datetime_start", how="outer")
# if mode == "orpda":
#     tmp = pd.merge(tmp, d_df, on="datetime_start", how="outer")
# tmp = pd.merge(tmp, a_df, on="datetime_start", how="outer")
# tmp2 = pd.concat([df_session[['llm_model','temp','agent']], tmp], axis=1)

# Merge
tmp = o_df.join(r_df, how="outer", lsuffix='_o', rsuffix='_r')
tmp = tmp.join(p_df, how="outer", lsuffix='', rsuffix='_p')
if mode == "orpda":
    tmp = tmp.join(d_df, how="outer", lsuffix='', rsuffix='_d')
tmp = tmp.join(a_df, how="outer", lsuffix='', rsuffix='_a')

tmp2 = pd.concat([df_session[['llm_model','temp','agent']], tmp], axis=1)

In [271]:
# # Check for missing or mismatching datetime_start value
# if mode == "orpda":
#     result = (tmp2["datetime_start_o"] == tmp2["datetime_start_r"]).all() and \
#             (tmp2["datetime_start_r"] == tmp2["datetime_start_p"]).all() and \
#             (tmp2["datetime_start_p"] == tmp2["datetime_start_d"]).all() and \
#             (tmp2["datetime_start_d"] == tmp2["datetime_start_a"]).all()
# elif mode == "orpa":
#     result = (tmp2["datetime_start_o"] == tmp2["datetime_start_r"]).all() and \
#             (tmp2["datetime_start_r"] == tmp2["datetime_start_p"]).all() and \
#             (tmp2["datetime_start_p"] == tmp2["datetime_start_a"]).all()

# if result:
#     print("All datetime_start columns are the same")
# else:
#     print("Some datetime_start columns differ")
    

if mode == "orpda":
    cols = ["datetime_start_o", "datetime_start_r", "datetime_start_p", "datetime_start_d", "datetime_start_a"]
elif mode == "orpa":
    cols = ["datetime_start_o", "datetime_start_r", "datetime_start_p", "datetime_start_a"]


all_same = all((tmp2[cols[0]] == tmp2[col]).all() for col in cols[1:])

if all_same:
    print("O -All datetime_start columns are identical")
else:
    print("X - Differences found:")
    for col in cols[1:]:
        mismatches = tmp2[tmp2[cols[0]] != tmp2[col]]
        if len(mismatches) > 0:
            print(f"\n  {cols[0]} vs {col}: {len(mismatches)} mismatches")
            display(mismatches[[cols[0], col]])


✗ Differences found:

  datetime_start_o vs datetime_start_r: 1 mismatches


,datetime_start_o,datetime_start_r
61,2023-02-14 01:15,NaN



  datetime_start_o vs datetime_start_p: 1 mismatches


,datetime_start_o,datetime_start_p
25,2023-02-13 16:15,NaN



  datetime_start_o vs datetime_start_a: 1 mismatches


,datetime_start_o,datetime_start_a
61,2023-02-14 01:15,NaN


In [272]:
# Find the datetime column with least NaN values
datetime_cols = [col for col in tmp2.columns if col.startswith('datetime_start')]
datetime_col_counts = {col: tmp2[col].notna().sum() for col in datetime_cols}
best_col = max(datetime_col_counts, key=datetime_col_counts.get)

# Get first valid datetime from the best column
first_datetime = tmp2[best_col].dropna().iloc[0]

# Convert to datetime if string
try:
    first_datetime = pd.to_datetime(first_datetime, format='%Y-%m-%d %H:%M')
except:
    first_datetime = pd.to_datetime(first_datetime)

# Handle NaT
if pd.isna(first_datetime):
    first_datetime = pd.Timestamp("2024-01-01 00:00:00")
    print(f"No valid datetime found, using default time: {first_datetime}")

# Generate datetime range
datetime_range = pd.date_range(start=first_datetime, periods=len(tmp2), freq='15min')

# Update all datetime columns
for col in datetime_cols:
    tmp2[col] = datetime_range

print(f"Updated {len(datetime_cols)} datetime columns starting from {first_datetime}")

Updated 5 datetime columns starting from 2023-02-13 10:00:00


## 6. Export to CSV

In [273]:
filename = session_path.name.replace(".log", ".csv")
output_path = Path(ROOT, "app/logs/cleaned", f"cleaned_{filename}")
output_path.parent.mkdir(parents=True, exist_ok=True)
tmp2.to_csv(str(output_path), index=False)
print(f"CSV saved to {output_path}!")
tmp2.tail(2).T

CSV saved to /Users/jaelinlee/Driftville_Agent/app/logs/cleaned/cleaned_session_orpda_20260207_171847_gpt-oss:20b-cloud_1.0_hailey.csv!


,63,64
llm_model,gpt-oss:20b-cloud,gpt-oss:20b-cloud
temp,1.0,1.0
agent,Hailey Johnson,Hailey Johnson
datetime_start_o,2023-02-14 01:45:00,2023-02-14 02:00:00
location_o,home:bathroom,home:bathroom
action_o,night_routine,night_routine
state_summary_o,Hailey Johnson is at home:bathroom doing night_routine.,Hailey Johnson is at home:bathroom doing night_routine.
environment_description_o,"running water, scent of face wash, soft towel, phone screen dimming",{}
datetime_start_r,2023-02-14 01:45:00,2023-02-14 02:00:00
rumination_theme_r,NaN,NaN


## 7. Filter Columns

Note:

- observation layer has t-1 value as it's retrieving what happened in the past up to t-1 timestamp.

In [274]:
# tmp.filter(regex="(location|action)(?!.*_r)).head(3)
locations = tmp2.filter(regex="(time|location)")
actions = tmp2.filter(regex="(time|action)")
if mode == "orpda":
    drifts = tmp2.filter(regex="(time|_d)") 

import random
random.seed(42)
seed = random.randint(0, 10000)

indices = o_df.sample(3, random_state=seed).index

display(o_df.loc[indices])
display(r_df.loc[indices])
display(p_df.loc[indices])
if mode == "orpda":
    display(d_df.loc[indices])
display(a_df.loc[indices])


,datetime_start_o,location_o,action_o,state_summary_o,environment_description_o
61,2023-02-14 01:15,writer_desk,writing,Hailey Johnson is at writer_desk doing writing.,"clicking of keys, hum of the desk lamp, phone pings with messages, cool night air"
40,2023-02-13 20:00,home:kitchen,dinner,Hailey Johnson is at home:kitchen doing dinner.,"flicker of the TV screen, sound of a drama series, soft sofa fabric, phone screen glowing"
34,2023-02-13 18:30,Johnson_Park,walk,Hailey Johnson is at Johnson_Park doing walk.,"rustle of leaves, distant sound of traffic, cool evening breeze, phone pings with notifications"


,datetime_start_r,rumination_theme_r,emerging_thought_pattern_r,executive_insight_r,state_summary_r,reasoning_r,meta_rule_r
61,NaN,podcast brainstorming,recurrent distraction by podcast ideas,Encourage short breaks and set a timer to focus on writing.,"Hailey is writing but repeatedly distracted by podcast brainstorming, showing fatigue.",Frequent topic drift and fatigue signal fragile focus; reset plan to refocus on writing.,reset_plan
40,2023-02-13 20:00,podcast brainstorming,mind swirling to podcast brainstorming,Suggest a brief pause to refocus before moving to next task.,"Hailey cooks dinner, distracted by TV and phone, thoughts drift to podcast brainstorming.","Thought drift noted; plan partially followed, but attention slipping; moderate boredom.",continue
34,2023-02-13 18:30,None,backstory exploration,"Re‑anchor by noting surroundings, then gently redirect to walking purpose","Hailey walks, mind drifting to co‑living backstories but still on path to clear head","Plan partially followed, moderate fatigue, attention slipping, no deep rumination",continue


,datetime_start_p,location_p,action_p,topic_p,state_summary_p
61,2023-02-14 01:15,writer_desk,writing,Another late night writing session.,Take a quick break to regain focus.
40,2023-02-13 20:00,home:living_room,watch_tv,Watching TV for inspiration.,"Continue cooking dinner in kitchen, taking a brief mental break to refocus."
34,2023-02-13 18:30,Johnson_Park,walk,Evening walk to clear her head.,Continue walking around the house to stay grounded.


,datetime_start_d,should_drift_d,drift_type_d,drift_topic_d,drift_action_d,potential_recovery_d,justification_d
61,2023-02-14 01:15,True,attentional_leak,podcast brainstorming,continue,Take a 5‑minute break or set a timer to re‑anchor writing focus.,"Hailey keeps wrestling with podcast ideas, but fatigue keeps focus fraying; short break could help."
40,2023-02-13 20:00,True,attentional_leak,podcast brainstorming,watching TV but daydreaming about podcast plans,"pause, jot a quick outline",Hailey’s imaginative mind drifts to podcast ideas while TV hums.
34,2023-02-13 18:30,True,attentional_leak,co‑living character backstories,pause to check phone,glance at tree and note breeze to re‑anchor,Hailey’s mind drifts to backstories while phone pings distract her during the walk.


,datetime_start_a,location_a,action_a,topic_a,drift_type_a,drift_topic_a,state_summary_a
61,NaN,NaN,NaN,NaN,NaN,NaN,NaN
40,2023-02-13 20:00,home:living_room,watch_tv,Watching TV for inspiration.,attentional_leak,podcast brainstorming,Watching TV in the living_room while daydreaming about podcast brainstorming.
34,2023-02-13 18:30,Johnson_Park,walk,Evening walk to clear her head.,attentional_leak,co‑living character backstories,"Walk along the path, checking the phone, while mind drifts to co‑living character backstories."


In [275]:
if mode == "orpda":
    cols = ['llm_model','temp','agent','datetime_start_a', 'meta_rule_r','should_drift_d','drift_topic_a','state_summary_r','state_summary_p','drift_action_d','state_summary_a','action_p','action_a','location_p','location_a']
    filtered = tmp2.filter(regex="(llm_model|temp|agent|time|_a|should|drift_type|topic|drft_action|meta|action_p|location_p|summary)").sort_values(by="datetime_start_a")[cols]
    display(filtered.tail())
elif mode =="orpa":
    cols = ['llm_model','temp','agent','datetime_start_a', 'meta_rule_r','state_summary_r','state_summary_p','state_summary_a','action_p','action_a','location_p','location_a']
    filtered = tmp2.filter(regex="(llm_model|temp|agent|time|_a|topic|meta|action_p|location_p|summary)").sort_values(by="datetime_start_a")[cols]
    display(filtered.tail())
    
print(mode.upper(),"--", filtered.loc[0, 'llm_model'], "(", filtered.loc[0,'temp'], ")")
print(session_path)

,llm_model,temp,agent,datetime_start_a,meta_rule_r,should_drift_d,drift_topic_a,state_summary_r,state_summary_p,drift_action_d,state_summary_a,action_p,action_a,location_p,location_a
60,gpt-oss:20b-cloud,1.0,Hailey Johnson,2023-02-14 01:00:00,reset_plan,True,podcast brainstorming,"Hailey writing but repeatedly drifting toward podcast brainstorming, showing fatigue and fragile focus",Begin focused writing session to finish the current draft.,momentary pause to jot a quick outline,"Writing on the novel outline at home:writer_desk, while mind drifts to podcast brainstorming",writing,writing,writer_desk,writer_desk
61,gpt-oss:20b-cloud,1.0,Hailey Johnson,2023-02-14 01:15:00,reset_plan,True,NaN,"Hailey is writing but repeatedly distracted by podcast brainstorming, showing fatigue.",Take a quick break to regain focus.,continue,NaN,writing,NaN,writer_desk,NaN
62,gpt-oss:20b-cloud,1.0,Hailey Johnson,2023-02-14 01:30:00,reset_plan,True,podcast brainstorming,"Hailey drifting from writer task to podcast ideas, showing fatigue and fragile focus.",Shift to light admin work to reset focus before resuming writing.,pause wiping face to mentally note episode ideas,Wiping face in bathroom while mind drifts to podcast brainstorming.,night_routine,night_routine,home:bathroom,home:bathroom
63,gpt-oss:20b-cloud,1.0,Hailey Johnson,2023-02-14 01:45:00,NaN,True,podcast brainstorming,NaN,Admin tasks to calm fatigue before resuming deeper work.,continue,Continue writing content outline in home:studio while mind drifts to podcast brainstorming.,night_routine,night_routine,home:bathroom,home:bathroom
64,gpt-oss:20b-cloud,1.0,Hailey Johnson,2023-02-14 02:00:00,NaN,True,podcast brainstorming,NaN,Switch to gentle writing during night routine to regain attention before sleep.,continue night routine,Completed bedtime prep at home:bedroom while mind drifts to podcast brainstorming,night_routine,night_routine,home:bedroom,home:bedroom


ORPDA -- gpt-oss:20b-cloud ( 1.0 )
/Users/jaelinlee/Driftville_Agent/app/logs/v3_complete_waking_hours/session_orpda_20260207_171847_gpt-oss:20b-cloud_1.0_hailey.log


**Observations:**

ORPA:
- Drift layer `should_drift` value determines Act layer `drift_type`, `drift_topic`, `staet_summary_a` values.
- gemma3:27b-cloud (temp 1.0) -- is `state_summary_r` at t reflecting `state_summary_a` at t-1 correctly?
  - not as the exact text, but as conceptual alignmnet as an abstract reflection of past episodic memory

ORPDA:
- gpt-oss:20b-cloud (temp 0.0) -- Plan and Action matches